# Assignment 10: MLOps Workflow Simulation

**Course:** Tata Technologies TechPulse Applied AI/ML  
**Topic:** Machine Learning Operations (MLOps) Lifecycle  
**Components:** MLflow Experiment Tracking, Model Artifact Management, Docker Containerization, CI/CD Pipeline  

--- 

## 1. Aim and Objectives
- Simulate an end-to-end educational **MLOps Workflow** for automotive machine learning models.
- Train a Random Forest regression model to predict vehicle resale price.
- Track hyperparameter configurations, metrics, and trained model artifacts using **MLflow**.
- Save trained model artifact (`rf_model.joblib`) for autonomous inference deployment.
- Define containerization setup (`Dockerfile`, `requirements.txt`, `predict.py`) that loads the exported model artifact.
- Design a GitHub Actions CI/CD pipeline (`.github/workflows/mlops.yml`) for automated build, test, and inference verification.

## 2. Import Libraries & Set Local MLflow Tracking URI
Import MLflow, Scikit-Learn, Joblib, Pandas, NumPy, and Matplotlib.

In [1]:
import os
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Allow file store and set sqlite tracking backend for portability
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
mlflow.set_tracking_uri('sqlite:///mlflow.db')

# Set reproducibility seed
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"MLflow Version: {mlflow.__version__}")

C:\Users\smkan\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/09/29 00:20:14 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\smkan\AppData\Roaming\Python\Python313\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


MLflow Tracking URI: sqlite:///mlflow.db
MLflow Version: 3.16.1


## 3. Generate Automotive Training Dataset
Create a dataset of 500 vehicle records with numerical features (`age_years`, `mileage_km`, `engine_size_L`, `horsepower`) and target `price`.

In [2]:
n_samples = 500
age_years = np.random.uniform(1, 15, n_samples)
mileage_km = age_years * np.random.uniform(10000, 18000, n_samples)
engine_size_L = np.random.uniform(1.0, 4.0, n_samples)
horsepower = engine_size_L * np.random.uniform(60, 90, n_samples)

price = 35000 - (age_years * 1800) - (mileage_km * 0.08) + (horsepower * 80) + np.random.normal(0, 1000, n_samples)
price = np.clip(price, 3000, 80000)

df = pd.DataFrame({
    'age_years': np.round(age_years, 1),
    'mileage_km': np.round(mileage_km, 0),
    'engine_size_L': np.round(engine_size_L, 2),
    'horsepower': np.round(horsepower, 1),
    'price': np.round(price, 2)
})

X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED)
print(f"Dataset generated successfully. X_train: {X_train.shape}, X_test: {X_test.shape}")

Dataset generated successfully. X_train: (400, 4), X_test: (100, 4)


## 4. MLflow Experiment Tracking & Model Export
Create MLflow experiment `Automotive_Price_Prediction_Experiment`, train `RandomForestRegressor`, log MLflow run metrics, and export `rf_model.joblib` artifact.

In [3]:
experiment_name = "Automotive_Price_Prediction_Experiment"
mlflow.set_experiment(experiment_name)

n_estimators = 200
max_depth = 12

with mlflow.start_run(run_name="Random_Forest_Resale_Price_Run") as run:
    run_id = run.info.run_id
    exp_id = run.info.experiment_id
    
    # Train model
    model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=SEED)
    model.fit(X_train, y_train)
    
    # Predict and evaluate
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Log parameters to MLflow
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("random_state", SEED)
    
    # Log metrics to MLflow
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2_Score", r2)
    
    # Log scikit-learn model artifact using cloudpickle
    mlflow.sklearn.log_model(model, name="model", serialization_format="cloudpickle")
    
    # Export model artifact for container deployment
    model_export_path = 'rf_model.joblib'
    joblib.dump(model, model_export_path)
    print(f"Exported trained model artifact to '{model_export_path}'")
    
    print(f"==================================================")
    print(f"MLflow Run Completed Successfully!")
    print(f"  Experiment Name: {experiment_name}")
    print(f"  Experiment ID:   {exp_id}")
    print(f"  Run ID:          {run_id}")
    print(f"==================================================")
    print(f"Logged Metrics:")
    print(f"  MAE:  ${mae:.2f}")
    print(f"  RMSE: ${rmse:.2f}")
    print(f"  R²:   {r2:.4f}")

2026/09/29 00:20:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Exported trained model artifact to 'rf_model.joblib'
MLflow Run Completed Successfully!
  Experiment Name: Automotive_Price_Prediction_Experiment
  Experiment ID:   1
  Run ID:          9bc392aa04874444808e93d5f046d616
Logged Metrics:
  MAE:  $1382.36
  RMSE: $1761.26
  R²:   0.9836


## 5. Load Exported Model & Verify Real Model Inference
Load `rf_model.joblib` and execute actual model inference on a new sample vehicle input.

In [4]:
loaded_rf_model = joblib.load('rf_model.joblib')

sample_vehicle = pd.DataFrame([{
    'age_years': 3.0,
    'mileage_km': 35000.0,
    'engine_size_L': 2.0,
    'horsepower': 180.0
}])

predicted_price = loaded_rf_model.predict(sample_vehicle)[0]

print(f"Loaded Model File: 'rf_model.joblib'")
print(f"Sample Input Profile:\n{sample_vehicle.to_string(index=False)}")
print(f"\nActual Model Predicted Resale Price: ${predicted_price:.2f}")

Loaded Model File: 'rf_model.joblib'
Sample Input Profile:
 age_years  mileage_km  engine_size_L  horsepower
       3.0     35000.0            2.0       180.0

Actual Model Predicted Resale Price: $40808.93


## 6. Standalone Inference Script (`predict.py`) Local Verification
Verify that `predict.py` executes standalone loading `rf_model.joblib`.

In [5]:
import subprocess
res = subprocess.run(['python', 'predict.py'], capture_output=True, text=True)
print("--- STANDALONE PREDICT.PY STDOUT ---")
print(res.stdout)

--- STANDALONE PREDICT.PY STDOUT ---
Automotive MLOps Model Inference Service Running
Successfully loaded trained model artifact from 'rf_model.joblib'.

Sample Input Features:
 age_years  mileage_km  engine_size_L  horsepower
       3.0     35000.0            2.0       180.0

Actual Model Predicted Resale Price: $40808.93
Inference service execution completed successfully.



## 7. Containerization Setup & CI/CD Workflow
Inspect `Dockerfile`, `requirements.txt`, and `.github/workflows/mlops.yml`.

In [6]:
if os.path.exists('Dockerfile'):
    with open('Dockerfile', 'r') as f:
        print("--- DOCKERFILE CONTENT ---")
        print(f.read())

cicd_path = os.path.join('..', '..', '.github', 'workflows', 'mlops.yml')
if os.path.exists(cicd_path):
    with open(cicd_path, 'r') as f:
        print("--- GITHUB ACTIONS CI/CD WORKFLOW (.github/workflows/mlops.yml) ---")
        print(f.read())

--- DOCKERFILE CONTENT ---
# Minimal Dockerfile for Automotive MLOps Model Inference Service
FROM python:3.11-slim

WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy model artifact and inference script
COPY rf_model.joblib .
COPY predict.py .

# Run inference service entrypoint
CMD ["python", "predict.py"]

--- GITHUB ACTIONS CI/CD WORKFLOW (.github/workflows/mlops.yml) ---
name: Automotive MLOps CI/CD Pipeline

on:
  push:
    branches: [ main, master ]
  pull_request:
    branches: [ main, master ]

jobs:
  mlops-pipeline:
    runs-on: ubuntu-latest

    steps:
    - name: Checkout Code
      uses: actions/checkout@v3

    - name: Set up Python 3.11
      uses: actions/setup-python@v4
      with:
        python-version: '3.11'

    - name: Install Dependencies
      run: |
        python -m pip install --upgrade pip
        pip install -r Assignments/Assignment_10/requirements.txt

    - name: Execute Model Trainin

## 8. MLOps Lifecycle Summary & Conclusion
1. **Experiment Tracking:** MLflow tracked hyperparameter parameters (`n_estimators=200`, `max_depth=12`), metrics (MAE, RMSE, $R^2$), and logged model artifacts.
2. **Exported Model Inference:** Exported model artifact (`rf_model.joblib`) was loaded by `predict.py` to run actual machine learning model inference without hardcoded formulas.
3. **Autonomous Containerization:** `Dockerfile` copies `rf_model.joblib` and `predict.py` for standalone container execution.
4. **CI/CD Pipeline:** GitHub Actions workflow (`mlops.yml`) automates notebook execution, MLflow logging, and `predict.py` inference checks.